# 03_gold_transform

Silver -> Gold (dimensional model for BI).

| Table | Type | Grain | Load strategy |
|---|---|---|---|
| `gold.dim_symbol` | Dimension | 1 row per trading pair | MERGE (tracks status changes / delistings) |
| `gold.dim_date` | Dimension | 1 row per calendar day | Overwrite (tiny) |
| `gold.fact_daily_ohlcv` | Fact | symbol x day | MERGE on (symbol_key, date_key) |
| `gold.volatility_daily` | Derived fact | symbol x day | Overwrite from fact |
| `gold.moving_averages` | Derived fact | symbol x day | Overwrite from fact |
| `gold.symbol_correlation` | Derived fact | symbol pair x day | Overwrite from fact |
| `cryptolens.pipeline_execution_logs` | Audit (Phase 2 table) | Gold steps logged with layer = 'Silver-to-Gold' | Append |

Run this as the **last task** of the daily job, after `02_silver_transform`.

## 0. Config  (EDIT THIS CELL IF YOUR NAMES DIFFER)

In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable
from contextlib import contextmanager
from datetime import datetime, timezone
import uuid

spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG = "workspace"
SILVER_TABLE = "workspace.cryptolens.silver_klines"      # <-- confirm your Silver table name
GOLD_SCHEMA = f"{CATALOG}.gold"
EXCHANGE_INFO_DIR = "/Volumes/workspace/cryptolens/bronze_raw/arqumumais@gmail.com/exchange_info"

# Map: canonical name -> your actual Silver column name  (<-- confirm these)
SILVER_COLS = {
    "symbol": "symbol",
    "open_time": "open_time",              # TIMESTAMP of candle open
    "open": "open",
    "high": "high",
    "low": "low",
    "close": "close",
    "volume": "volume",                    # base asset volume
    "quote_volume": "quote_asset_volume",  # USDT volume
    "trades": "number_of_trades",
}

FULL_REFRESH = False        # True = recompute every day from Silver
LOOKBACK_DAYS = 2           # incremental: reprocess the last N days to absorb late/corrected data

# Indicator settings. Only ~1 month of data is loaded, so 50/200-day windows are not
# computable yet. Switch to 50 / 200 once enough history exists.
MA_SHORT, MA_LONG = 5, 20
VOL_WINDOW = 7
CORR_WINDOW = 14
CORR_TOP_N = 15

RUN_ID = str(uuid.uuid4())
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
print("Run:", RUN_ID)

## 1. Audit logging (writes into the Phase 2 `pipeline_execution_logs` table, layer = 'Silver-to-Gold')

In [0]:
LOG_TABLE = "workspace.cryptolens.pipeline_execution_logs"
LOG_SCHEMA = spark.table(LOG_TABLE).schema      # reuse the exact schema Bronze/Silver write with

RUN_LOAD_TYPE = ("full" if FULL_REFRESH or not spark.catalog.tableExists(f"{GOLD_SCHEMA}.fact_daily_ohlcv")
                 else "incremental")

def _merge_metrics(table_name):
    """Rows the latest MERGE really inserted/updated, from Delta commit metrics."""
    h = DeltaTable.forName(spark, table_name).history(1).select("operation", "operationMetrics").collect()[0]
    if h["operation"] != "MERGE":
        return None, None
    m = h["operationMetrics"]
    return int(m.get("numTargetRowsInserted", 0)), int(m.get("numTargetRowsUpdated", 0))

def _log(table_name, operation, rows_read, rows_written, start, status,
         message=None, inserted=None, updated=None):
    end = datetime.now(timezone.utc).replace(tzinfo=None)
    row = [(str(uuid.uuid4()), "Silver-to-Gold",
            f"{table_name} [{operation}] source_rows={int(rows_read)}",
            RUN_LOAD_TYPE, start, end, status,
            int(inserted if inserted is not None else rows_written),
            int(updated or 0),
            message[:500] if message else None)]
    spark.createDataFrame(row, schema=LOG_SCHEMA).write.format("delta").mode("append").saveAsTable(LOG_TABLE)

@contextmanager
def tracked(table_name, operation):
    """Usage: with tracked('x','MERGE') as t: ...; t['read']=n; t['written']=m"""
    start = datetime.now(timezone.utc).replace(tzinfo=None)
    t = {"read": 0, "written": 0}
    try:
        yield t
        ins, upd = (None, None)
        if operation == "MERGE":
            ins, upd = _merge_metrics(table_name)
        _log(table_name, operation, t["read"], t["written"], start, "SUCCESS", inserted=ins, updated=upd)
        print(f"[OK] {table_name} ({operation}): read={t['read']:,} written={t['written']:,}")
    except Exception as e:
        _log(table_name, operation, t["read"], t["written"], start, "FAILURE", message=str(e))
        raise

def table_exists(name):
    return spark.catalog.tableExists(name)

## 2. Read Silver and validate columns

In [0]:
silver_raw = spark.table(SILVER_TABLE)
missing = [v for v in SILVER_COLS.values() if v not in silver_raw.columns]
if missing:
    print("Silver columns found:", silver_raw.columns)
    raise ValueError(f"Columns not found in {SILVER_TABLE}: {missing}. Fix SILVER_COLS in the config cell.")

silver = (silver_raw
    .select(*[F.col(actual).alias(canon) for canon, actual in SILVER_COLS.items()])
    # Silver divides the raw epoch by 1000 (assumes milliseconds). Binance 2026 files are in
    # MICROseconds, so Silver timestamps land ~year 58,000. Undo it here: true_seconds = silver_seconds / 1000.
    # Rows already in a sane range (genuine ms files) are left alone.
    .withColumn("open_time",
        F.when(F.col("open_time") > F.lit("2200-01-01").cast("timestamp"),
               F.round(F.col("open_time").cast("double") / 1000).cast("long").cast("timestamp"))
         .otherwise(F.col("open_time")))
    .withColumn("trade_date", F.to_date("open_time")))

FACT = f"{GOLD_SCHEMA}.fact_daily_ohlcv"

if not FULL_REFRESH and table_exists(FACT):
    last = spark.table(FACT).agg(F.max("trade_date")).first()[0]
    if last is not None:
        silver = silver.filter(F.col("trade_date") >= F.date_sub(F.lit(last), LOOKBACK_DAYS))
        print(f"Incremental mode: processing Silver from {last} minus {LOOKBACK_DAYS} days")
else:
    print("Full build from Silver")

## 3. dim_symbol  (from the exchangeInfo snapshot, with status change tracking)

In [0]:
DIM_SYMBOL = f"{GOLD_SCHEMA}.dim_symbol"

files = sorted([f.path for f in dbutils.fs.ls(EXCHANGE_INFO_DIR) if f.name.endswith(".json")])
latest = files[-1]
snapshot_date = latest.rsplit("exchangeInfo-", 1)[1].replace(".json", "")
print("Using snapshot:", latest, "->", snapshot_date)

with tracked(DIM_SYMBOL, "MERGE") as t:
    info = (spark.read.option("multiLine", True).json(latest)
            .select(F.explode("symbols").alias("s")).select("s.*"))
    src = (info.filter(F.col("quoteAsset") == "USDT")
        .select(
            F.abs(F.xxhash64("symbol")).alias("symbol_key"),
            F.col("symbol"),
            F.col("baseAsset").alias("base_asset"),
            F.col("quoteAsset").alias("quote_asset"),
            F.col("status"),
            F.col("isSpotTradingAllowed").alias("is_spot_trading_allowed"))
        .withColumn("snapshot_date", F.to_date(F.lit(snapshot_date))))
    t["read"] = src.count()

    if not table_exists(DIM_SYMBOL):
        (src.withColumn("previous_status", F.lit(None).cast("string"))
            .withColumn("status_changed_date", F.col("snapshot_date"))
            .withColumn("first_seen_date", F.col("snapshot_date"))
            .withColumn("last_seen_date", F.col("snapshot_date"))
            .drop("snapshot_date")
            .write.format("delta").saveAsTable(DIM_SYMBOL))
        t["written"] = t["read"]
    else:
        (DeltaTable.forName(spark, DIM_SYMBOL).alias("t")
            .merge(src.alias("s"), "t.symbol_key = s.symbol_key")
            .whenMatchedUpdate(condition="t.status <> s.status", set={
                "previous_status": "t.status", "status": "s.status",
                "status_changed_date": "s.snapshot_date", "last_seen_date": "s.snapshot_date",
                "is_spot_trading_allowed": "s.is_spot_trading_allowed"})
            .whenMatchedUpdate(set={"last_seen_date": "s.snapshot_date",
                                    "is_spot_trading_allowed": "s.is_spot_trading_allowed"})
            .whenNotMatchedInsert(values={
                "symbol_key": "s.symbol_key", "symbol": "s.symbol", "base_asset": "s.base_asset",
                "quote_asset": "s.quote_asset", "status": "s.status",
                "is_spot_trading_allowed": "s.is_spot_trading_allowed",
                "previous_status": "null", "status_changed_date": "s.snapshot_date",
                "first_seen_date": "s.snapshot_date", "last_seen_date": "s.snapshot_date"})
            .whenNotMatchedBySourceUpdate(condition="t.status <> 'DELISTED'", set={
                "previous_status": "t.status", "status": "'DELISTED'",
                "status_changed_date": f"DATE'{snapshot_date}'"})
            .execute())
        t["written"] = t["read"]

## 4. fact_daily_ohlcv  (1-minute Silver -> daily; MERGE keeps it idempotent)

In [0]:
with tracked(FACT, "MERGE") as t:
    t["read"] = silver.count()
    daily = (silver.groupBy("symbol", "trade_date").agg(
            F.min_by("open", "open_time").alias("open_price"),
            F.max("high").alias("high_price"),
            F.min("low").alias("low_price"),
            F.max_by("close", "open_time").alias("close_price"),
            F.sum("volume").alias("volume"),
            F.sum("quote_volume").alias("quote_volume"),
            F.sum("trades").alias("trade_count"),
            F.count("*").alias("candle_count"))
        .withColumn("symbol_key", F.abs(F.xxhash64("symbol")))
        .withColumn("date_key", F.date_format("trade_date", "yyyyMMdd").cast("int"))
        .withColumn("vwap", F.expr("try_divide(quote_volume, volume)"))
        .withColumn("_loaded_at", F.current_timestamp())
        .select("symbol_key", "date_key", "symbol", "trade_date", "open_price", "high_price",
                "low_price", "close_price", "volume", "quote_volume", "vwap",
                "trade_count", "candle_count", "_loaded_at"))
    t["written"] = daily.count()

    if not table_exists(FACT):
        daily.write.format("delta").saveAsTable(FACT)
    else:
        (DeltaTable.forName(spark, FACT).alias("t")
            .merge(daily.alias("s"), "t.symbol_key = s.symbol_key AND t.date_key = s.date_key")
            .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

## 5. dim_date

In [0]:
DIM_DATE = f"{GOLD_SCHEMA}.dim_date"
with tracked(DIM_DATE, "OVERWRITE") as t:
    bounds = spark.table(FACT).agg(F.min("trade_date"), F.max("trade_date")).first()
    d = (spark.sql(f"SELECT explode(sequence(DATE'{bounds[0]}', DATE'{bounds[1]}', interval 1 day)) AS full_date")
        .select(
            F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
            "full_date",
            F.year("full_date").alias("year"),
            F.quarter("full_date").alias("quarter"),
            F.month("full_date").alias("month"),
            F.date_format("full_date", "MMMM").alias("month_name"),
            F.dayofmonth("full_date").alias("day"),
            F.dayofweek("full_date").alias("day_of_week"),
            F.date_format("full_date", "EEEE").alias("day_name"),
            F.weekofyear("full_date").alias("week_of_year"),
            F.dayofweek("full_date").isin(1, 7).alias("is_weekend")))
    t["read"] = t["written"] = d.count()
    d.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DIM_DATE)

## 6. volatility_daily

In [0]:
VOL = f"{GOLD_SCHEMA}.volatility_daily"
with tracked(VOL, "OVERWRITE") as t:
    f = spark.table(FACT)
    t["read"] = f.count()
    w = Window.partitionBy("symbol_key").orderBy("trade_date")
    wv = w.rowsBetween(-(VOL_WINDOW - 1), 0)
    v = (f.withColumn("prev_close", F.lag("close_price").over(w))
        .withColumn("daily_return", F.expr("try_divide(close_price, prev_close) - 1"))
        .withColumn("log_return", F.expr("ln(try_divide(close_price, prev_close))"))
        .withColumn("intraday_range_pct", F.expr("try_divide(high_price - low_price, open_price)"))
        .withColumn("rolling_vol", F.stddev("log_return").over(wv))
        .withColumn("rolling_vol_annualized", F.col("rolling_vol") * F.sqrt(F.lit(365.0)))
        .select("symbol_key", "date_key", "symbol", "trade_date", "close_price",
                "daily_return", "log_return", "intraday_range_pct", "rolling_vol", "rolling_vol_annualized"))
    t["written"] = v.count()
    v.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(VOL)

## 7. moving_averages  (golden / death cross)

In [0]:
MA = f"{GOLD_SCHEMA}.moving_averages"
with tracked(MA, "OVERWRITE") as t:
    f = spark.table(FACT)
    t["read"] = f.count()
    w = Window.partitionBy("symbol_key").orderBy("trade_date")
    rn = F.row_number().over(w)
    m = (f.withColumn("rn", rn)
        .withColumn("ma_short", F.when(rn >= MA_SHORT, F.avg("close_price").over(w.rowsBetween(-(MA_SHORT - 1), 0))))
        .withColumn("ma_long", F.when(rn >= MA_LONG, F.avg("close_price").over(w.rowsBetween(-(MA_LONG - 1), 0))))
        .withColumn("prev_short", F.lag("ma_short").over(w))
        .withColumn("prev_long", F.lag("ma_long").over(w))
        .withColumn("cross_signal",
            F.when((F.col("prev_short") <= F.col("prev_long")) & (F.col("ma_short") > F.col("ma_long")), "GOLDEN_CROSS")
             .when((F.col("prev_short") >= F.col("prev_long")) & (F.col("ma_short") < F.col("ma_long")), "DEATH_CROSS"))
        .withColumn("trend",
            F.when(F.col("ma_short") > F.col("ma_long"), "BULLISH")
             .when(F.col("ma_short") < F.col("ma_long"), "BEARISH"))
        .select("symbol_key", "date_key", "symbol", "trade_date", "close_price",
                "ma_short", "ma_long", "cross_signal", "trend")
        .withColumn("ma_short_window", F.lit(MA_SHORT))
        .withColumn("ma_long_window", F.lit(MA_LONG)))
    t["written"] = m.count()
    m.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(MA)

## 8. symbol_correlation  (rolling, top-N symbols by volume)

In [0]:
CORR = f"{GOLD_SCHEMA}.symbol_correlation"
with tracked(CORR, "OVERWRITE") as t:
    returns = spark.table(VOL).filter(F.col("log_return").isNotNull())
    top = (spark.table(FACT).groupBy("symbol").agg(F.sum("quote_volume").alias("v"))
           .orderBy(F.desc("v")).limit(CORR_TOP_N))
    r = returns.join(top.select("symbol"), "symbol").select("symbol", "trade_date", "log_return")
    t["read"] = r.count()

    a, b = r.alias("a"), r.alias("b")
    pairs = (a.join(b, (F.col("a.trade_date") == F.col("b.trade_date")) & (F.col("a.symbol") < F.col("b.symbol")))
        .select(F.col("a.symbol").alias("symbol_a"), F.col("b.symbol").alias("symbol_b"),
                F.col("a.trade_date").alias("trade_date"),
                F.col("a.log_return").alias("ret_a"), F.col("b.log_return").alias("ret_b")))
    w = Window.partitionBy("symbol_a", "symbol_b").orderBy("trade_date").rowsBetween(-(CORR_WINDOW - 1), 0)
    c = (pairs.withColumn("rolling_corr", F.corr("ret_a", "ret_b").over(w))
        .withColumn("obs", F.count("ret_a").over(w))
        .filter(F.col("obs") >= CORR_WINDOW)
        .withColumn("symbol_a_key", F.abs(F.xxhash64("symbol_a")))
        .withColumn("symbol_b_key", F.abs(F.xxhash64("symbol_b")))
        .withColumn("date_key", F.date_format("trade_date", "yyyyMMdd").cast("int"))
        .select("symbol_a_key", "symbol_b_key", "date_key", "symbol_a", "symbol_b",
                "trade_date", "rolling_corr"))
    t["written"] = c.count()
    c.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(CORR)

## 9. Read optimisation  (OPTIMIZE + Z-ORDER, Gold only)

In [0]:
for tbl, cols in [(FACT, "symbol_key, date_key"), (VOL, "symbol_key, date_key"),
                  (MA, "symbol_key, date_key"), (CORR, "symbol_a_key, date_key")]:
    with tracked(tbl, "OPTIMIZE") as t:
        spark.sql(f"OPTIMIZE {tbl} ZORDER BY ({cols})")

## 10. Verification

In [0]:
for tbl in ["dim_symbol", "dim_date", "fact_daily_ohlcv", "volatility_daily", "moving_averages", "symbol_correlation"]:
    print(f"{tbl:24s} {spark.table(f'{GOLD_SCHEMA}.{tbl}').count():>10,}")

orphans = (spark.table(FACT).join(spark.table(DIM_SYMBOL), "symbol_key", "left_anti").count())
print("fact rows with no dim_symbol match:", orphans)

display(spark.table(LOG_TABLE).orderBy(F.desc(LOG_SCHEMA.names[4])).limit(15))   # newest rows, incl. this gold run

## 11. Audit check (Gold steps in the shared log table)

In [0]:
%sql
SELECT layer, parameter_processed, load_type, status, rows_inserted, rows_updated, start_time, end_time
FROM workspace.cryptolens.pipeline_execution_logs
WHERE layer = 'Silver-to-Gold'
ORDER BY start_time DESC

In [0]:
%sql
DESCRIBE DETAIL workspace.cryptolens.bronze_klines  